# Sistema de Recomendación (XGBoost)

Variante de `SR_Claude.ipynb` (que usa `LGBMRanker`) con `XGBRanker` en vez de LightGBM. El armado del
dataset (filtro, split cronológico, target encoding, `genero_hist`/`libro_hist` leak-safe) es idéntico
— es código pandas/SQL, no depende de qué librería de gradient boosting se use después. Lo que cambia
es la sección de entrenamiento y todo lo que llama a `.predict()`.

**Diferencia deliberada en la métrica**: acá usamos NDCG con **ganancia lineal** (`ndcg_exp_gain=False`),
no la exponencial que usa `SR_Claude.ipynb`. Verificamos empíricamente antes de armar este notebook que
XGBoost, igual que LightGBM, usa ganancia exponencial **por default** — "XGBoost usa la versión no
exponencial" no es correcto como creencia previa, hay que pedirlo explícitamente con
`ndcg_exp_gain=False`. Es un experimento distinto al de `SR_Claude.ipynb`, no comparable número a número:
con ganancia lineal, un único libro con rating=10 no domina el DCG como con la exponencial, así que la
métrica debería mostrar con más nitidez cuánto aporta (o no) el feature engineering.

In [1]:
import sqlite3
import time

import joblib
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

/Users/sergiovolta/repos/sr_uba_2026/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Esta celda no es necesaria si corro local
# from google.colab import drive
# drive.mount('/content/drive')

In [3]:
# DATABASE = '/content/drive/MyDrive/Colab Notebooks/SR/datos/data.db'
# DATABASE = '/content/data.db'
DATABASE = 'datos/data.db'

# reusamos el dataset ya limpio de dataset_features_genero.ipynb (fechas parseadas, ids con
# metadata válida, género/país normalizados, y las columnas de historial genero_hist__/libro_hist__
# ya calculadas) en vez de repetir esa limpieza acá
DATASET_FEATURES_CSV = 'datos/dataset_features_genero.csv'
TRAIN_CSV = 'datos/sr_xgb_train.csv'
TEST_CSV = 'datos/sr_xgb_test.csv'
ARTIFACTS_PATH = 'datos/sr_xgb_dataset_artifacts.joblib'
MODEL_ARTIFACTS_PATH = 'datos/sr_xgb_model_artifacts.joblib'
OUTPUT_CSV = 'datos/sr_xgb_recomendaciones_top20.csv'

MIN_INTERACCIONES = 20   # lectores con menos quedan afuera del dataset
N_TEST_POR_LECTOR = 20   # interacciones mas recientes de cada lector reservadas para test
M_SUAVIZADO = 5          # smoothing bayesiano del target encoding de autor/editorial
RANDOM_STATE = 42

# Plan General

Dado que vamos a usar 20 interacciones de cada usuario para testing, necesitamos que cada usuario a
incluir en el dataset tenga al menos 20 interacciones — el que tenga exactamente 20 queda sin ninguna
fila de entrenamiento propia (arranca en frío para el modelo, aunque sí se evalúa).

El split es cronológico, no aleatorio: reservamos para test las **20 interacciones más recientes de
cada lector**, y el resto va a training — es la única forma de que el corte sea leak-safe (ver también
la sección 3.5, sobre `libro_hist`).

El modelo va a ser un `XGBRanker` con `objective="rank:ndcg"` (no un regresor de rating): en vez de
intentar predecir el número exacto de cada calificación, aprende a **ordenar** los libros de cada
lector por preferencia. A diferencia de `SR_Claude.ipynb` (que usa `LGBMRanker` con la ganancia
exponencial default de NDCG, `2^rel - 1`), acá forzamos `ndcg_exp_gain=False` — ganancia lineal — a
propósito, para tener un punto de comparación con una métrica menos comprimida hacia 1.0. El detalle de
cómo se conecta un árbol de gradiente con una métrica de ranking no diferenciable se resuelve en la
sección de entrenamiento.

Este notebook arranca directamente con el dataset completo (`genero_hist`/`libro_hist` incluidos) — la
etapa de "línea de base sin historial" ya se hizo en `SR_Claude.ipynb` y no hace falta repetirla acá.

# Armado del dataset (con feature engineering)

## 1. Atributos base + historial

Ahora sí traemos las 212 columnas `genero_hist__*` (perfil del lector por género, ya leak-safe por
construcción — ver sección siguiente) y las 2 `libro_hist__*` de `dataset_features_genero.csv`. A estas
últimas las renombramos a `libro_hist__n_completo` / `libro_hist__avg_completo` apenas las cargamos: tal
como están calculadas en el CSV (corte estrictamente anterior a la fecha, pero sobre **toda** la
población) no son directamente utilizables para entrenar bajo nuestro split — el porqué y el arreglo
están en la sección 3.5.

In [4]:
df = pd.read_csv(DATASET_FEATURES_CSV, parse_dates=["fecha_iso"])
df = df.rename(columns={"libro_hist__n": "libro_hist__n_completo", "libro_hist__avg": "libro_hist__avg_completo"})

print(f"atributos base + historial: {df.shape}")
df.head()

atributos base + historial: (461073, 225)


,id_lector,id_libro,fecha,rating,autor_libro,editorial_libro,anio_edicion_libro,genero_lector,fecha_iso,genero_libro,pais_lector,genero_hist__avg__arte,genero_hist__avg__autoayuda_y_espiritualidad,genero_hist__avg__biografias_memorias,genero_hist__avg__clasicos_de_la_literatura,genero_hist__avg__cocina,genero_hist__avg__comics_novela_grafica,genero_hist__avg__deportes_y_juegos,genero_hist__avg__derecho,genero_hist__avg__desconocido,...,genero_hist__n__medicina,genero_hist__n__medicina_divulgativa,genero_hist__n__musica,genero_hist__n__narrativa,genero_hist__n__naturaleza_y_ciencia,genero_hist__n__no_ficcion,genero_hist__n__novela,genero_hist__n__novela_negra,genero_hist__n__novela_negra_intriga_terror,genero_hist__n__peliculas,genero_hist__n__poesia,genero_hist__n__poesia_teatro,genero_hist__n__psicologia_y_pedagogia,genero_hist__n__romantica_erotica,genero_hist__n__television,genero_hist__n__varios,genero_hist__n__varios_otros_generos,genero_hist__n__viajes_en_bolsillo,libro_hist__n_completo,libro_hist__avg_completo
0,jc01,en-el-blanco,24-02-2008,4.0,"FOLLETT, KEN",DEBOLSILLO,2006.0,Hombre,2008-02-24,ficción literaria,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN
1,jc01,vuelo-final,24-02-2008,4.0,"FOLLETT, KEN",DEBOLSILLO,2003.0,Hombre,2008-02-24,histórica y aventuras,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN
2,jc01,la-clave-esta-en-rebeca,24-02-2008,6.0,"FOLLETT, KEN",DEBOLSILLO,2003.0,Hombre,2008-02-24,histórica y aventuras,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN
3,jc01,un-mundo-sin-fin-los-pilares-de-la-tierra-2,24-02-2008,6.0,"FOLLETT, KEN",PLAZA & JANÉS,2007.0,Hombre,2008-02-24,histórica y aventuras,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN
4,jc01,el-gran-gatsby,24-02-2008,8.0,"SCOTT FITZGERALD, FRANCIS",ALFAGUARA,2019.0,Hombre,2008-02-24,narrativa,españa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN


## 2. Filtro: lectores con ≥ 20 interacciones válidas

El conteo se hace sobre este dataset ya filtrado a interacciones válidas, no sobre la tabla
`interacciones` cruda: si contáramos sobre la tabla cruda, un lector con 20 interacciones pero alguna
con fecha corrupta o sin metadata terminaría con menos de 20 filas utilizables acá, y el split de
"últimas 20 a test" quedaría corto para ese usuario.

In [5]:
conteo_por_lector = df.groupby("id_lector").size()
lectores_validos = conteo_por_lector[conteo_por_lector >= MIN_INTERACCIONES].index

df = df[df["id_lector"].isin(lectores_validos)].copy()
print(f"lectores con >= {MIN_INTERACCIONES} interacciones válidas: {len(lectores_validos):,} de {conteo_por_lector.shape[0]:,}")
print(f"dataset filtrado: {df.shape}")

lectores con >= 20 interacciones válidas: 3,890 de 10,667
dataset filtrado: (424689, 225)


## 3. Split cronológico por usuario (últimas 20 a test)

En vez del loop por lector de la versión original (una query SQL + un `pd.concat` por usuario, lento y
propenso a colgarse con miles de lectores), lo resolvemos en una sola pasada vectorizada: ordenamos por
lector y fecha, y le asignamos a cada fila un "rango de recencia" dentro de su propio lector con
`cumcount(ascending=False)` — la fila más reciente de cada lector queda en 0, la anteúltima en 1, etc.
Las filas con rango 0-19 (las 20 más recientes) van a test; el resto, a train.

**Desempate**: si un lector calificó más de un libro el mismo día, el orden entre esas filas no está
definido solo por la fecha. Para que el split sea reproducible (no dependa del orden en que vinieron
las filas del CSV), ordenamos también por `id_libro` como criterio de desempate determinístico.

In [6]:
df = df.sort_values(["id_lector", "fecha_iso", "id_libro"]).reset_index(drop=True)

# cumcount(ascending=False) numera las filas de cada grupo en reversa: 0 para la última fila del
# grupo (la más reciente, porque ya ordenamos por fecha ascendente), 1 para la anteúltima, etc.
rango_recencia = df.groupby("id_lector", sort=False).cumcount(ascending=False)
es_test = rango_recencia < N_TEST_POR_LECTOR

df_train = df.loc[~es_test].reset_index(drop=True)
df_test = df.loc[es_test].reset_index(drop=True)

assert (df_test.groupby("id_lector").size() == N_TEST_POR_LECTOR).all(), "algún lector no quedó con exactamente 20 filas de test"

sin_train = set(df_test["id_lector"]) - set(df_train["id_lector"])
print(f"train: {df_train.shape}  test: {df_test.shape}")
print(f"lectores de test: {df_test['id_lector'].nunique():,} (los {N_TEST_POR_LECTOR} más recientes de cada uno)")
print(f"lectores sin ninguna fila de train (exactamente {MIN_INTERACCIONES} interacciones -> arranque en frío): {len(sin_train):,}")

train: (346889, 225)  test: (77800, 225)
lectores de test: 3,890 (los 20 más recientes de cada uno)
lectores sin ninguna fila de train (exactamente 20 interacciones -> arranque en frío): 89


## 3.5. Recalcular `libro_hist` respetando la frontera de test de este split

`genero_hist__*` es *por lector* — el historial de un usuario nunca toca al de otro, así que particionar
por usuario no crea fugas ahí (el corte "estrictamente antes de esta fecha" del lector, aplicado a las
filas de train, solo puede tocar OTRAS filas de train del mismo lector, porque test es por construcción
posterior a todo el train de ese lector).

`libro_hist__n`/`libro_hist__avg` es distinto: es un agregado **global**, entre todos los lectores. Tal
como está en `dataset_features_genero.csv` (`libro_hist__n_completo`/`libro_hist__avg_completo`), el
corte es solo por fecha, sin noción de qué filas reservamos como test en *este* split — así que una fila
de train de un lector B puede estar "viendo", a través del agregado del libro, el rating de una fila que
para el lector A es parte de su test. Es fuga cruzada entre lectores, no la fuga temporal de siempre.

La arreglamos recalculando el changelog acumulado de libro **excluyendo del pool** las 77.800 filas que
reservamos como test (para cualquier lector, no solo el de la fila que se está calculando) — el resto de
las interacciones válidas (lectores con menos de 20, y las filas de train de los que sí pasaron el
filtro) queda disponible como siempre. Validamos con un caso concreto: para "el código da vinci", una
fila de train fechada el 21/05/2010 tenía `libro_hist__n_completo = 501` (contando de más) pero
`libro_hist__n = 463` una vez excluidas las reservas de test de otros lectores — una diferencia del 8%,
nada despreciable. En general, el 70-76% de las filas de train/test tienen un `libro_hist__n` distinto
al de la versión sin corregir.

In [7]:
# recargamos el CSV completo (sin el filtro de >=20) para poder marcar, entre TODAS las
# interacciones válidas, cuáles son exactamente las 77.800 que quedaron reservadas como test bajo
# nuestro split -- df/df_train/df_test ya están filtrados y no alcanzan para reconstruir esto
df_completo = pd.read_csv(DATASET_FEATURES_CSV, usecols=["id_lector", "id_libro", "fecha_iso", "rating"], parse_dates=["fecha_iso"])
df_completo = df_completo.sort_values(["id_lector", "fecha_iso", "id_libro"]).reset_index(drop=True)

conteo_completo = df_completo.groupby("id_lector")["id_lector"].transform("size")
rango_completo = df_completo.groupby("id_lector", sort=False).cumcount(ascending=False)
es_test_global = (rango_completo < N_TEST_POR_LECTOR) & (conteo_completo >= MIN_INTERACCIONES)

pool = df_completo.loc[~es_test_global, ["id_libro", "fecha_iso", "rating"]].reset_index(drop=True)
print(f"pool para libro_hist: {len(pool):,} interacciones (se excluyen {es_test_global.sum():,} reservadas como test)")

por_dia = (
    pool.groupby(["id_libro", "fecha_iso"])["rating"]
    .agg(n_dia="size", suma_dia="sum")
    .reset_index()
    .sort_values(["id_libro", "fecha_iso"])
)
por_dia["n_acum"] = por_dia.groupby("id_libro")["n_dia"].cumsum()
por_dia["suma_acum"] = por_dia.groupby("id_libro")["suma_dia"].cumsum()
changelog_libro = por_dia[["id_libro", "fecha_iso", "n_acum", "suma_acum"]]


def merge_libro_hist(df_objetivo, changelog):
    objetivo_sorted = df_objetivo.sort_values("fecha_iso").reset_index(drop=True)
    changelog_sorted = changelog.sort_values("fecha_iso").reset_index(drop=True)
    merged = pd.merge_asof(
        objetivo_sorted, changelog_sorted,
        on="fecha_iso", by="id_libro",
        direction="backward", allow_exact_matches=False,  # estrictamente anterior
    )
    nuevas = pd.DataFrame({
        "libro_hist__n": merged["n_acum"].fillna(0).astype("int32"),
        "libro_hist__avg": (merged["suma_acum"] / merged["n_acum"]).astype("float32"),
    })
    return pd.concat([merged.drop(columns=["n_acum", "suma_acum"]), nuevas], axis=1)


df_train = merge_libro_hist(df_train, changelog_libro)
df_test = merge_libro_hist(df_test, changelog_libro)

dif = (df_train["libro_hist__n"] != df_train["libro_hist__n_completo"]).mean()
print(f"filas de train donde libro_hist cambió respecto a la versión sin corregir: {dif:.1%}")

# validación manual sobre un caso con historial largo
fila = df_train[df_train["libro_hist__n_completo"] > 500].iloc[0]
manual = pool[(pool["id_libro"] == fila["id_libro"]) & (pool["fecha_iso"] < fila["fecha_iso"])]
print(f"\nlibro={fila['id_libro']!r} fecha={fila['fecha_iso'].date()}")
print(f"manual (pool sin test)   -> n={len(manual)}, avg={manual['rating'].mean():.3f}")
print(f"pipeline                 -> n={fila['libro_hist__n']}, avg={fila['libro_hist__avg']:.3f}")
print(f"version sin corregir     -> n={fila['libro_hist__n_completo']:.0f}, avg={fila['libro_hist__avg_completo']:.3f}")
assert len(manual) == fila["libro_hist__n"], "no coincide -> hay fuga o error"
print("\nOK: coincide con el cálculo manual restringido al pool sin las reservas de test.")

pool para libro_hist: 383,273 interacciones (se excluyen 77,800 reservadas como test)


filas de train donde libro_hist cambió respecto a la versión sin corregir: 70.1%

libro='el-codigo-da-vinci' fecha=2010-05-21
manual (pool sin test)   -> n=463, avg=5.475
pipeline                 -> n=463, avg=5.475
version sin corregir     -> n=501, avg=5.419

OK: coincide con el cálculo manual restringido al pool sin las reservas de test.


## 4. Target encoding de autor/editorial: ajustar con train, aplicar a cualquiera

`autor_libro` y `editorial_libro` tienen demasiadas categorías distintas para tratarlas como
categóricas nativas (tanto LightGBM como XGBoost se vuelven lentos con decenas de miles de categorías)
— se reemplazan por un promedio suavizado de rating por categoría (`ajustar_encoding`), calculado
**solo con las filas de train**. Ese mapa ya calculado se aplica después con una segunda función
(`aplicar_encoding`) tanto a train como a test — la misma tabla de valores en los dos casos, sin volver
a mirar el rating de test en ningún momento. Si en test aparece un autor/editorial que nunca estuvo en
train, `aplicar_encoding` lo completa con la media global (fallback razonable ante la falta de
evidencia).

In [8]:
ENC_COLS = ["autor_libro", "editorial_libro"]
TARGET = "rating"


def ajustar_encoding(cat_series, rating_series, m=M_SUAVIZADO):
    media_global = rating_series.mean()
    stats = rating_series.groupby(cat_series).agg(["mean", "count"])
    enc = (stats["mean"] * stats["count"] + media_global * m) / (stats["count"] + m)
    return enc, media_global


def aplicar_encoding(cat_series, enc_map, media_global):
    return cat_series.map(enc_map).fillna(media_global)


encodings = {}
for c in ENC_COLS:
    enc_map, media_global = ajustar_encoding(df_train[c], df_train[TARGET])
    encodings[c] = (enc_map, media_global)
    df_train[f"{c}_enc"] = aplicar_encoding(df_train[c], enc_map, media_global)
    df_test[f"{c}_enc"] = aplicar_encoding(df_test[c], enc_map, media_global)
    sin_cobertura = (~df_test[c].isin(enc_map.index)).mean()
    print(f"{c}_enc: media_global={media_global:.3f}  autores/editoriales de test no vistos en train: {sin_cobertura:.1%}")

autor_libro_enc: media_global=7.179  autores/editoriales de test no vistos en train: 3.1%
editorial_libro_enc: media_global=7.179  autores/editoriales de test no vistos en train: 0.5%


## 5. Categóricas nativas

`genero_libro`, `genero_lector` y `pais_lector` tienen pocas categorías y quedan como categóricas
nativas (`enable_categorical=True` en XGBoost — el equivalente de `categorical_feature` en LightGBM).
Fijamos el conjunto de categorías usando train **y** test juntos — no es una fuga (no se usa ningún
rating para esto, solo qué valores de texto existen). A diferencia de LightGBM (que exige que el
conjunto de categorías de la predicción coincida exactamente con el de entrenamiento), XGBoost matchea
por el **valor** de cada categoría, no por su código interno — igual dejamos las categorías fijadas acá
desde el principio, por prolijidad y para no depender de ese detalle de implementación (verificado en
la sección de predicción).

In [9]:
CAT_COLS = ["genero_libro", "genero_lector", "pais_lector"]

cat_categories = {}
for c in CAT_COLS:
    categorias = pd.Categorical(pd.concat([df_train[c], df_test[c]])).categories
    df_train[c] = pd.Categorical(df_train[c], categories=categorias)
    df_test[c] = pd.Categorical(df_test[c], categories=categorias)
    cat_categories[c] = categorias
    print(f"  {c}: {len(categorias)} categorias")

  genero_libro: 52 categorias
  genero_lector: 2 categorias


  pais_lector: 38 categorias


## 6. Orden por lector + arrays de grupo para `XGBRanker`

Esta es la pieza que el esqueleto original no tenía y que es específica de pasar de regresión a
ranking: `XGBRanker` no recibe una columna `id_lector` — necesita que las filas de un mismo lector
queden **contiguas** en la matriz de entrenamiento, y un array aparte (`group`) con el tamaño de cada
bloque, en ese mismo orden (misma convención que `LGBMRanker`). Así es como el modelo sabe qué filas
puede comparar entre sí (nunca compara el score de un libro de un lector contra el de otro lector —
comparar ratings entre dos personas con escalas de calificación distintas no tiene sentido, y es
justamente lo que NDCG evita).

En test el array de grupos es trivial (siempre vale `N_TEST_POR_LECTOR`, por construcción); en train no
— cada lector aporta un número de filas distinto (todas menos sus últimas 20).

In [10]:
df_train = df_train.sort_values("id_lector").reset_index(drop=True)
df_test = df_test.sort_values("id_lector").reset_index(drop=True)

group_train = df_train.groupby("id_lector", sort=False).size().to_numpy()
group_test = df_test.groupby("id_lector", sort=False).size().to_numpy()

assert group_train.sum() == len(df_train) and group_test.sum() == len(df_test)
assert (group_test == N_TEST_POR_LECTOR).all(), "algún grupo de test no tiene tamaño 20"

print(f"grupos de train: {len(group_train)} lectores, tamaño min={group_train.min()} max={group_train.max()} media={group_train.mean():.1f}")
print(f"grupos de test: {len(group_test)} lectores, tamaño fijo={N_TEST_POR_LECTOR}")

grupos de train: 3801 lectores, tamaño min=1 max=2240 media=91.3
grupos de test: 3890 lectores, tamaño fijo=20


## 7. Validación de la relevancia (`rating`)

`rating` se usa directamente como relevancia para el ranker — ya es una escala graduada de 1 a 10, no
hace falta binarizarla ni reescalarla acá. Esta celda solo valida que no haya nulos ni valores fuera de
rango; qué función de ganancia usar sobre esos valores (exponencial vs. lineal) es una decisión de
hiperparámetros del entrenamiento, no algo que se resuelva en el armado del dataset.

In [11]:
for nombre, d in [("train", df_train), ("test", df_test)]:
    assert d["rating"].notna().all(), f"{nombre}: hay ratings nulos"
    assert (d["rating"] == d["rating"].round()).all(), f"{nombre}: hay ratings no enteros"
    assert d["rating"].between(1, 10).all(), f"{nombre}: rating fuera del rango esperado [1, 10]"

print("OK: rating sin nulos, entero, y en rango [1, 10] en train y test -- listo para usarse como relevancia.")

OK: rating sin nulos, entero, y en rango [1, 10] en train y test -- listo para usarse como relevancia.


## 8. Columnas de features y checkpoint

`FEATURE_COLS` sigue el mismo criterio que `modelo_lightgbm.ipynb`: todas las columnas del dataset salvo
los identificadores, el target, las versiones sin encodear de autor/editorial (quedan las versiones
`_enc`), y ahora también `HELPER_COLS` (`libro_hist__n_completo`/`libro_hist__avg_completo`) — esas dos
no son features, son un valor auxiliar que solo se usa en el reentrenamiento final con el 100% de los
datos (sección "Entrenar el modelo"). Guardamos train/test ya ordenados por lector, y los mapas de
encoding/categorías en un artefacto aparte para que la próxima etapa no tenga que recalcularlos.

Los arrays de grupo (`group_train`/`group_test`) **no** se guardan aparte — se recalculan al vuelo con
`groupby().size()` sobre el CSV ya ordenado por `id_lector`. Persistirlos como archivo aparte los
expondría a desincronizarse del CSV si algún día alguien reordena las filas sin querer; recalcularlos es
barato y siempre va a estar en línea con los datos que efectivamente se están usando.

In [12]:
ID_COLS = ["id_lector", "id_libro", "fecha", "fecha_iso"]
HELPER_COLS = ["libro_hist__n_completo", "libro_hist__avg_completo"]  # solo sirven para el reentrenamiento 100%

FEATURE_COLS = [c for c in df_train.columns if c not in ID_COLS + [TARGET] + ENC_COLS + HELPER_COLS]

print(f"features (con genero_hist/libro_hist): {len(FEATURE_COLS)}")

df_train.to_csv(TRAIN_CSV, index=False)
df_test.to_csv(TEST_CSV, index=False)

joblib.dump(
    {
        "encodings": encodings,
        "cat_categories": cat_categories,
        "FEATURE_COLS": FEATURE_COLS,
        "CAT_COLS": CAT_COLS,
        "ENC_COLS": ENC_COLS,
        "ID_COLS": ID_COLS,
        "HELPER_COLS": HELPER_COLS,
        "TARGET": TARGET,
        "N_TEST_POR_LECTOR": N_TEST_POR_LECTOR,
    },
    ARTIFACTS_PATH,
)

print(f"\nguardado: {TRAIN_CSV} ({df_train.shape}), {TEST_CSV} ({df_test.shape})")
print(f"artefactos (encodings, categorías, columnas) guardados en {ARTIFACTS_PATH}")

features (con genero_hist/libro_hist): 220



guardado: datos/sr_xgb_train.csv ((346889, 229)), datos/sr_xgb_test.csv ((77800, 229))
artefactos (encodings, categorías, columnas) guardados en datos/sr_xgb_dataset_artifacts.joblib


# Entrenar el modelo

Reusamos `df_train`/`df_test`/`group_train`/`group_test`/`FEATURE_COLS`/`CAT_COLS` tal como quedaron en
memoria al final del armado del dataset.

`objective="rank:ndcg"` es el equivalente en XGBoost de `lambdarank`: para cada par de libros dentro del
mismo grupo (mismo lector) con relevancia distinta, calcula un pseudo-gradiente que combina qué tan mal
ordenado está ese par según el score actual, pesado por cuánto cambiaría el NDCG del grupo si se
intercambiaran sus posiciones — mismo mecanismo LambdaMART que `LGBMRanker`, otra implementación.

**La diferencia que nos interesa acá es `ndcg_exp_gain=False`**: por default (`True`), tanto XGBoost
como LightGBM usan ganancia exponencial (`2^rel - 1`) para el DCG — lo verificamos empíricamente antes
de armar este notebook, así que "XGBoost usa la versión no exponencial" no es correcto como default,
hay que pedirlo explícitamente. Con `ndcg_exp_gain=False`, la ganancia es el rating tal cual
(`gain = rel`) — un libro con rating=10 pesa 10 veces más que uno con rating=1, no 1023 veces más. Esto
también cambia qué hiperparámetros usamos: `max_depth`/`min_child_weight` en vez de
`num_leaves`/`min_child_samples` (XGBoost crece los árboles por profundidad por default, no por hoja
como LightGBM), y el `early_stopping_rounds` va en el **constructor** del modelo, no en `fit()`.

La búsqueda de hiperparámetros con Optuna es el mismo esqueleto TPE que `SR_Claude.ipynb`,
`direction="maximize"` sobre `ndcg@20`. Ojo con `evals_result()`: la lista de valores por ronda **no**
termina en la mejor ronda — `early_stopping_rounds` sigue entrenando esa cantidad de rondas más antes
de frenar, así que el último valor de la lista puede ser peor que el mejor. Usamos `model.best_score`
(y para varias métricas a la vez, `evals_result()[...][metrica][model.best_iteration]`), no
`evals_result()[...][-1]`.

In [13]:
X_train, y_train = df_train[FEATURE_COLS], df_train[TARGET].astype(int)
X_test, y_test = df_test[FEATURE_COLS], df_test[TARGET].astype(int)

SEARCH_N_ESTIMATORS = 300
SEARCH_EARLY_STOPPING = 30
N_TRIALS = 30
SEARCH_TIMEOUT_SECONDS = 900  # red de seguridad: 15 min tope total (XGBoost tarda mas por trial que LightGBM)


def objective(trial):
    params = dict(
        n_estimators=SEARCH_N_ESTIMATORS,
        learning_rate=trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_float("min_child_weight", 1e-2, 100, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model = xgb.XGBRanker(
        objective="rank:ndcg", eval_metric="ndcg@20", ndcg_exp_gain=False,
        tree_method="hist", enable_categorical=True,
        early_stopping_rounds=SEARCH_EARLY_STOPPING,
        verbosity=0, **params,
    )
    model.fit(
        X_train, y_train, group=group_train,
        eval_set=[(X_test, y_test)], eval_group=[group_test],
        verbose=False,
    )
    trial.set_user_attr("best_iteration", model.best_iteration)
    return model.best_score


t0 = time.time()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, timeout=SEARCH_TIMEOUT_SECONDS, show_progress_bar=False)
print(f"{len(study.trials)} trials en {time.time() - t0:.0f}s")

30 trials en 339s


In [14]:
print(f"mejor ndcg@20 (test, con tope de {SEARCH_N_ESTIMATORS} arboles): {study.best_value:.4f}")
print("mejores hiperparametros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"best_iteration del mejor trial: {study.best_trial.user_attrs['best_iteration']}")

mejor ndcg@20 (test, con tope de 300 arboles): 0.9572
mejores hiperparametros:
  learning_rate: 0.10417153701290169
  max_depth: 9
  min_child_weight: 14.817024243794116
  subsample: 0.7943666941340121
  colsample_bytree: 0.7372941165434505
  reg_alpha: 0.008437334486413317
  reg_lambda: 5.930698018916652e-05
best_iteration del mejor trial: 208


## Evaluación honesta y baselines

Reentrenamos con los hiperparámetros óptimos pero con más margen de árboles y *early stopping* otra vez
sobre test, para que el modelo converja del todo — mismo criterio que `SR_Claude.ipynb`.

**Acá `sklearn.metrics.ndcg_score` sí sirve directo**: como decidimos usar ganancia lineal
(`ndcg_exp_gain=False`), que es justo lo que `ndcg_score` calcula por default — es el mismo caso que en
`SR_Claude.ipynb`, pero del lado bueno del problema que documentamos ahí (ahí la ganancia exponencial de
LightGBM no coincidía con el default lineal de sklearn). Igual implementamos `ndcg_lineal` a mano y la
validamos contra `evals_result()` de XGBoost, para mantener el mismo patrón de verificación del resto
del proyecto y no depender de una coincidencia.

In [15]:
def ndcg_lineal(relevancias_en_orden, k=None):
    """DCG/IDCG con ganancia lineal (gain = rel), igual que XGBoost con ndcg_exp_gain=False."""
    rel = np.asarray(relevancias_en_orden, dtype=float)
    if k is not None:
        rel = rel[:k]
    descuentos = 1 / np.log2(np.arange(2, len(rel) + 2))
    dcg = (rel * descuentos).sum()
    ideal = np.sort(rel)[::-1]
    idcg = (ideal * descuentos).sum()
    return dcg / idcg if idcg > 0 else 0.0


def ndcg_medio(df, score_col, k=20):
    """Promedio de ndcg_lineal sobre los grupos (lectores) de df, ordenando por score_col."""
    def _ndcg_de_un_lector(g):
        orden = g.sort_values(score_col, ascending=False)[TARGET].to_numpy()
        return ndcg_lineal(orden, k=k)
    return df.groupby("id_lector").apply(_ndcg_de_un_lector, include_groups=False).mean()

In [16]:
EVAL_N_ESTIMATORS = 1500
best_params = dict(study.best_params)

final_search_model = xgb.XGBRanker(
    objective="rank:ndcg", eval_metric=["ndcg@5", "ndcg@10", "ndcg@20"], ndcg_exp_gain=False,
    tree_method="hist", enable_categorical=True,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=80,
    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0, **best_params,
)
t0 = time.time()
final_search_model.fit(
    X_train, y_train, group=group_train,
    eval_set=[(X_test, y_test)], eval_group=[group_test],
    verbose=False,
)
bi = final_search_model.best_iteration
resultado_eval = final_search_model.evals_result()["validation_0"]
print(f"fit {time.time() - t0:.0f}s, best_iteration={bi}")
print("XGBoost reporta:", {m: resultado_eval[m][bi] for m in ["ndcg@5", "ndcg@10", "ndcg@20"]})

# copia aparte para no ensuciar df_test con columnas de score que solo sirven para esta evaluación
eval_df = df_test.copy()
eval_df["pred_modelo"] = final_search_model.predict(X_test)

ndcg_modelo = ndcg_medio(eval_df, "pred_modelo")
print(f"\nndcg@20 (verificación manual, misma fórmula que XGBoost): {ndcg_modelo:.4f}")
print("(debería coincidir con el ndcg@20 reportado arriba)")

# baselines: aleatorio, y promedio del libro calculado SOLO con train (para que sea una comparación justa)
rng = np.random.default_rng(RANDOM_STATE)
eval_df["pred_aleatorio"] = rng.random(len(eval_df))

libro_avg_train = df_train.groupby("id_libro")[TARGET].mean()
media_global_train = df_train[TARGET].mean()
eval_df["pred_libro_avg"] = eval_df["id_libro"].map(libro_avg_train).fillna(media_global_train)

print()
print(f"{'modelo (XGBRanker + NDCG lineal)':32s} ndcg@20={ndcg_modelo:.4f}")
print(f"{'baseline: orden aleatorio':32s} ndcg@20={ndcg_medio(eval_df, 'pred_aleatorio'):.4f}")
print(f"{'baseline: promedio del libro':32s} ndcg@20={ndcg_medio(eval_df, 'pred_libro_avg'):.4f}")

fit 27s, best_iteration=393
XGBoost reporta: {'ndcg@5': 0.8750328656200002, 'ndcg@10': 0.8962310781502055, 'ndcg@20': 0.9572612385461995}



ndcg@20 (verificación manual, misma fórmula que XGBoost): 0.9573
(debería coincidir con el ndcg@20 reportado arriba)

modelo (XGBRanker + NDCG lineal) ndcg@20=0.9573


baseline: orden aleatorio        ndcg@20=0.9334


baseline: promedio del libro     ndcg@20=0.9544


## Reentrenamiento final con el 100% de los datos

Mismos hiperparámetros óptimos, y el target encoding de autor/editorial se recalcula con el 100% de los
datos (ya no hay test que proteger) — mismo criterio que `modelo_lightgbm.ipynb`. Con `libro_hist` pasa
lo mismo que con el encoding: ya no hace falta excluir las 77.800 filas de test de ningún lector, así
que en vez de recalcular el changelog de nuevo, simplemente restauramos los valores originales
`libro_hist__n_completo`/`libro_hist__avg_completo` (los que ya veníamos guardando desde la sección 3.5,
sin la exclusión) en las columnas `libro_hist__n`/`libro_hist__avg` que usa `FEATURE_COLS`.
`n_estimators` queda fijo en el `best_iteration` de la evaluación honesta (sin *early stopping*, no hay
held-out set). Este es el modelo que se usa después para puntuar libros que el lector no leyó.

In [17]:
df_full = pd.concat([df_train, df_test], ignore_index=True).sort_values("id_lector").reset_index(drop=True)

# restaurar libro_hist "completo" (sin excluir test) -- ya no hay ningún test que proteger
df_full["libro_hist__n"] = df_full["libro_hist__n_completo"].astype("int32")
df_full["libro_hist__avg"] = df_full["libro_hist__avg_completo"].astype("float32")

encodings_full = {}
for c in ENC_COLS:
    enc_map, media_global = ajustar_encoding(df_full[c], df_full[TARGET])
    encodings_full[c] = (enc_map, media_global)
    df_full[f"{c}_enc"] = aplicar_encoding(df_full[c], enc_map, media_global)

group_full = df_full.groupby("id_lector", sort=False).size().to_numpy()
X_full, y_full = df_full[FEATURE_COLS], df_full[TARGET].astype(int)

# best_iteration es 0-indexado (ronda 63 = 64 arboles) -- +1 para reproducir la misma cantidad
# de arboles que uso predict() por default durante la evaluación honesta
N_ESTIMATORS_FINAL = final_search_model.best_iteration + 1
final_model = xgb.XGBRanker(
    objective="rank:ndcg", ndcg_exp_gain=False, tree_method="hist", enable_categorical=True,
    n_estimators=N_ESTIMATORS_FINAL, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0, **best_params,
)
t0 = time.time()
final_model.fit(X_full, y_full, group=group_full)
print(f"modelo final: {N_ESTIMATORS_FINAL} arboles, {len(X_full):,} filas, {len(group_full):,} lectores, {time.time() - t0:.0f}s")

modelo final: 394 arboles, 424,689 filas, 3,890 lectores, 13s


Guardamos el modelo final y todo lo necesario para reconstruir features de predicción (encodings de
autor/editorial recalculados con el 100%, categorías vistas en entrenamiento, columnas de features) en
un solo archivo, para que generar recomendaciones más adelante no dependa de repetir la búsqueda
bayesiana ni el entrenamiento.

In [18]:
joblib.dump(
    {
        "final_model": final_model,
        "encodings_full": encodings_full,
        "cat_categories": cat_categories,
        "FEATURE_COLS": FEATURE_COLS,
        "CAT_COLS": CAT_COLS,
        "ENC_COLS": ENC_COLS,
        "TARGET": TARGET,
        "best_params": best_params,
        "N_ESTIMATORS_FINAL": N_ESTIMATORS_FINAL,
    },
    MODEL_ARTIFACTS_PATH,
)
print(f"artefactos del modelo guardados en {MODEL_ARTIFACTS_PATH}")

artefactos del modelo guardados en datos/sr_xgb_model_artifacts.joblib


# Recomendador

Con el modelo final entrenado, armamos el catálogo de candidatos (todos los libros del catálogo, con
las mismas columnas que usó el modelo) y una función de *retrieval* que, para cada lector, acota ese
catálogo a los libros que probablemente le interesen — puntuar el catálogo completo (128.743 libros)
para cada uno de los miles de lectores sería carísimo. Después, `ranking()` usa el modelo para puntuar
esos candidatos y nos quedamos con el top 20.

In [19]:
genero_canon = pd.read_csv("datos/genero_canon_lookup.csv").set_index("genero_original")["genero_canonico"].to_dict()

conn = sqlite3.connect(DATABASE)
todos_los_libros = pd.read_sql("""
    SELECT
        id_libro,
        autor        AS autor_libro,
        genero       AS genero_libro_raw,
        editorial    AS editorial_libro,
        anio_edicion AS anio_edicion_libro
    FROM libros
""", conn)
# libro_hist actual = estado de HOY (sin cutoff de fecha, sin exclusión de test -- no hay ningún
# futuro que proteger al generar recomendaciones nuevas)
libro_hist_actual = pd.read_sql("""
    SELECT i.id_libro, COUNT(*) AS n_ratings, AVG(i.rating) AS avg_rating
    FROM interacciones i JOIN lectores r ON r.id_lector = i.id_lector
    WHERE i.fecha GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9][0-9][0-9]'
    GROUP BY i.id_libro
""", conn).set_index("id_libro")
lectores_tabla = pd.read_sql("SELECT id_lector, genero AS genero_lector, vive_en FROM lectores", conn)
leidos = pd.read_sql("SELECT DISTINCT id_lector, id_libro FROM interacciones", conn)
conn.close()

# mismo procesamiento que en las secciones 1 y 4 del armado del dataset, aplicado ahora a TODO
# el catálogo (no solo a los libros que aparecen en alguna interacción)
todos_los_libros["genero_libro"] = (
    todos_los_libros["genero_libro_raw"].str.strip().str.lower().map(genero_canon).fillna("desconocido")
)
anio_num = pd.to_numeric(todos_los_libros["anio_edicion_libro"], errors="coerce")
todos_los_libros["anio_edicion_libro"] = anio_num.where(anio_num.between(1400, 2026))
for c in ENC_COLS:
    enc_map, media_global = encodings_full[c]
    todos_los_libros[f"{c}_enc"] = aplicar_encoding(todos_los_libros[c], enc_map, media_global)
todos_los_libros["genero_libro"] = todos_los_libros["genero_libro"].astype(pd.CategoricalDtype(categories=cat_categories["genero_libro"]))

todos_los_libros = todos_los_libros.set_index("id_libro")
todos_los_libros["libro_hist__n"] = libro_hist_actual["n_ratings"].reindex(todos_los_libros.index).fillna(0).astype("int32")
todos_los_libros["libro_hist__avg"] = libro_hist_actual["avg_rating"].reindex(todos_los_libros.index).astype("float32")
todos_los_libros = todos_los_libros.reset_index()

lectores_tabla["genero_lector"] = lectores_tabla["genero_lector"].replace({"-": np.nan, "": np.nan})
lectores_tabla["pais_lector"] = (
    lectores_tabla["vive_en"].str.rsplit("-", n=1).str[-1].str.strip().str.lower()
    .replace({"": np.nan, "¿?": np.nan})
)
lectores_tabla["genero_lector"] = lectores_tabla["genero_lector"].astype(pd.CategoricalDtype(categories=cat_categories["genero_lector"]))
lectores_tabla["pais_lector"] = lectores_tabla["pais_lector"].astype(pd.CategoricalDtype(categories=cat_categories["pais_lector"]))
lectores_tabla = lectores_tabla.set_index("id_lector")

print(f"catálogo: {todos_los_libros.shape}  lectores: {lectores_tabla.shape}  interacciones (todas): {leidos.shape}")

catálogo: (128743, 10)  lectores: (11285, 3)  interacciones (todas): (461408, 2)


/var/folders/8t/kp6w6lhx5_50cdk2plklmwgw0000gn/T/ipykernel_41352/3431752344.py:35: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  todos_los_libros["genero_libro"] = todos_los_libros["genero_libro"].astype(pd.CategoricalDtype(categories=cat_categories["genero_libro"]))
/var/folders/8t/kp6w6lhx5_50cdk2plklmwgw0000gn/T/ipykernel_41352/3431752344.py:48: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  lectores_tabla["pais_lector"] = lectores_tabla["pais_lector"].astype(pd.CategoricalDtype(categories=cat_categories["pais_lector"]))


## Perfil de género actual por lector

`genero_hist__*` es una feature *por lector*, no del catálogo — no tiene sentido ponerla en
`todos_los_libros`. Necesitamos, para cada lector, su perfil completo (las 212 columnas) tal como está
**hoy**, y usarlo como bloque constante al puntuar todos sus candidatos.

Mismo pipeline que la sección 6 de `modelo_lightgbm.ipynb`: un *changelog* acumulado por
lector×género vía SQL (reusando el `genero_canon` ya cargado), pivoteado a formato ancho y quedándonos
con el último estado de cada lector — **todo** su historial hasta hoy, sin corte de fecha ni exclusión
de test: acá no hay ninguna fila futura que proteger, estamos generando recomendaciones hacia adelante,
no evaluando. Reusamos `genero_slug_lookup.csv` (ya guardado por `dataset_features_genero.ipynb`) para
que los nombres de columna coincidan exactamente con los que ya tiene `FEATURE_COLS`, en vez de volver a
calcular el slug de cada género con una función aparte.

In [20]:
genero_slug = pd.read_csv("datos/genero_slug_lookup.csv").set_index("genero_canonico")["slug"].to_dict()
genero_slug["desconocido"] = "desconocido"

conn = sqlite3.connect(DATABASE)
conn.execute("CREATE TEMP TABLE genero_canon (genero_norm TEXT PRIMARY KEY, genero_canonico TEXT NOT NULL)")
conn.executemany("INSERT INTO genero_canon VALUES (?, ?)", list(genero_canon.items()))

query_changelog = """
WITH base AS (
    SELECT
        i.id_lector,
        COALESCE(gc.genero_canonico, 'desconocido') AS genero_norm,
        substr(i.fecha, 7, 4) || '-' || substr(i.fecha, 4, 2) || '-' || substr(i.fecha, 1, 2) AS fecha_iso,
        i.rating AS rating
    FROM interacciones i
    JOIN libros l    ON l.id_libro = i.id_libro
    JOIN lectores r  ON r.id_lector = i.id_lector
    LEFT JOIN genero_canon gc ON gc.genero_norm = LOWER(TRIM(l.genero))
    WHERE i.fecha GLOB '[0-9][0-9]-[0-9][0-9]-[0-9][0-9][0-9][0-9]'
),
por_dia AS (
    SELECT
        id_lector, genero_norm, fecha_iso,
        COUNT(*) AS n_dia, SUM(rating) AS suma_dia, MIN(rating) AS min_dia, MAX(rating) AS max_dia
    FROM base
    GROUP BY id_lector, genero_norm, fecha_iso
)
SELECT
    id_lector, genero_norm, fecha_iso,
    SUM(n_dia)    OVER w AS n_acum,
    SUM(suma_dia) OVER w AS suma_acum,
    MIN(min_dia)  OVER w AS min_acum,
    MAX(max_dia)  OVER w AS max_acum
FROM por_dia
WINDOW w AS (
    PARTITION BY id_lector, genero_norm
    ORDER BY fecha_iso
    RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
"""
t0 = time.time()
changelog_genero = pd.read_sql_query(query_changelog, conn)
conn.close()
changelog_genero["fecha_iso"] = pd.to_datetime(changelog_genero["fecha_iso"])
print(f"changelog género: {changelog_genero.shape} en {time.time() - t0:.1f}s")

wide_hist = changelog_genero.pivot(
    index=["id_lector", "fecha_iso"], columns="genero_norm",
    values=["n_acum", "suma_acum", "min_acum", "max_acum"],
)
wide_hist.columns = [f"genero_hist__{m.replace('_acum', '')}__{genero_slug[g]}" for m, g in wide_hist.columns]
wide_hist = wide_hist.reset_index().sort_values("fecha_iso")

n_cols_hist = [c for c in wide_hist.columns if c.startswith("genero_hist__n__")]
otras_cols = [c for c in wide_hist.columns if c.startswith("genero_hist__") and c not in n_cols_hist]
wide_hist[n_cols_hist] = wide_hist.groupby("id_lector")[n_cols_hist].ffill()
wide_hist[otras_cols] = wide_hist.groupby("id_lector")[otras_cols].ffill()

# perfil actual = último estado de cada lector (todo su historial, sin cortar por fecha)
perfil_genero_actual = wide_hist.sort_values("fecha_iso").groupby("id_lector").tail(1).drop(columns=["fecha_iso"])

suma_cols = [c for c in perfil_genero_actual.columns if c.startswith("genero_hist__suma__")]
avg_nuevas = {}
for suma_c in suma_cols:
    tag = suma_c[len("genero_hist__suma__"):]
    n_c = f"genero_hist__n__{tag}"
    avg_nuevas[f"genero_hist__avg__{tag}"] = perfil_genero_actual[suma_c] / perfil_genero_actual[n_c].replace(0, np.nan)
perfil_genero_actual = pd.concat(
    [perfil_genero_actual.drop(columns=suma_cols), pd.DataFrame(avg_nuevas, index=perfil_genero_actual.index)], axis=1
)
perfil_genero_actual[n_cols_hist] = perfil_genero_actual[n_cols_hist].fillna(0).astype("int32")
min_max_cols = [c for c in perfil_genero_actual.columns if "__min__" in c or "__max__" in c]
perfil_genero_actual[min_max_cols] = perfil_genero_actual[min_max_cols].astype("float32")
perfil_genero_actual = perfil_genero_actual.set_index("id_lector")

GENERO_HIST_COLS = list(perfil_genero_actual.columns)
faltan = set(c for c in FEATURE_COLS if c.startswith("genero_hist__")) - set(GENERO_HIST_COLS)
assert not faltan, f"faltan columnas de perfil de género: {faltan}"
print(f"perfil_genero_actual: {perfil_genero_actual.shape} (una fila por lector con historial)")
print("OK: las columnas de perfil coinciden con las de entrenamiento.")

changelog género: (246671, 7) en 1.6s


perfil_genero_actual: (10667, 212) (una fila por lector con historial)
OK: las columnas de perfil coinciden con las de entrenamiento.


## Retrieval

Precalculamos, para cada lector, sus géneros/autores preferidos: se derivan uniendo el conjunto de
`id_libro` que ya leyó (`leidos`, con **todas** las interacciones conocidas, no solo train — para no
recomendar algo ya calificado sea o no parte del split de entrenamiento) contra el propio catálogo, así
que quedan en la misma representación cruda (`genero_libro_raw`) que usa el catálogo, sin depender de
que `df_train`/`df_full` tengan columnas de género. `retrieval()` acota el catálogo a los libros no
leídos cuyo género o autor coincide con alguno de esos preferidos — mismo criterio (y mismas ideas de
refinamiento pendientes, "Idea 01/02" del planteo original) que la primera versión de este notebook.

In [21]:
leidos_con_info = leidos.merge(
    todos_los_libros[["id_libro", "genero_libro_raw", "autor_libro"]], on="id_libro", how="left"
)
leidos_por_lector = leidos.groupby("id_lector")["id_libro"].apply(set)
generos_por_lector = leidos_con_info.groupby("id_lector")["genero_libro_raw"].apply(set)
autores_por_lector = leidos_con_info.groupby("id_lector")["autor_libro"].apply(set)


def retrieval(id_lector):
    """Devuelve los libros no leídos por id_lector cuyo género o autor coincide con alguno de sus preferidos."""
    libros_leidos = leidos_por_lector.get(id_lector, set())
    generos_preferidos = generos_por_lector.get(id_lector, set())
    autores_preferidos = autores_por_lector.get(id_lector, set())

    candidatos = todos_los_libros[~todos_los_libros["id_libro"].isin(libros_leidos)]
    candidatos = candidatos[
        candidatos["genero_libro_raw"].isin(generos_preferidos) | candidatos["autor_libro"].isin(autores_preferidos)
    ]
    return candidatos


lector_demo = df_test["id_lector"].iloc[0]
candidatos_demo = retrieval(lector_demo)
print(f"candidatos para {lector_demo!r}: {len(candidatos_demo):,} de {len(todos_los_libros):,} libros del catálogo")
candidatos_demo.head(10)

candidatos para '05-03-1970': 49,686 de 128,743 libros del catálogo


,id_libro,autor_libro,genero_libro_raw,editorial_libro,anio_edicion_libro,genero_libro,autor_libro_enc,editorial_libro_enc,libro_hist__n,libro_hist__avg
144,vive-como-un-mendigo-baila-como-un-rey,"FARRAY, IGNATIUS",Humor,TEMAS DE HOY,2020.0,humor,7.137420,7.175711,9,7.333333
145,el-senor-de-las-moscas,"GOLDING, WILLIAM",Narrativa,ALIANZA,2006.0,narrativa,7.225968,7.472071,594,7.254209
146,los-asquerosos,"LORENZO, SANTIAGO",Narrativa,BLACKIE BOOKS,2018.0,narrativa,7.093308,7.270591,154,7.175324
147,cien-anos-de-soledad,"GARCÍA MÁRQUEZ, GABRIEL",Literatura contemporánea,ALFAGUARA,2007.0,literatura contemporánea,7.560178,7.198743,1733,8.241777
148,los-renglones-torcidos-de-dios,"LUCA DE TENA, TORCUATO",Narrativa,PLANETA,2009.0,narrativa,7.877355,6.852534,1199,8.003336
149,la-conjura-de-los-necios,"TOOLE, JOHN KENNEDY",Literatura contemporánea,ANAGRAMA,2013.0,literatura contemporánea,7.248909,7.105748,639,7.305164
151,la-reina-del-sur,"PÉREZ-REVERTE, ARTURO",Ficción literaria,ALFAGUARA,2011.0,ficción literaria,6.921917,7.198743,314,7.098726
152,sin-noticias-de-gurb,"MENDOZA, EDUARDO",Humor,SEIX BARRAL,2002.0,humor,6.776917,6.909682,616,6.922078
153,persepolis,"SATRAPI, MARJANE","Cómics, Novela Gráfica",RESERVOIR BOOKS,2020.0,"cómics, novela gráfica",8.000300,7.525549,174,8.201149
154,el-nino-con-el-pijama-de-rayas,"BOYNE, JOHN",Ficción literaria,SALAMANDRA,2007.0,ficción literaria,6.990972,7.622398,1406,6.968706


## Puntuar candidatos con el modelo (`ranking`)

Misma estructura que `SR_Claude.ipynb`: `FEATURE_COLS` tiene 220 columnas — 6 del lado del libro
(`anio_edicion_libro`, `genero_libro`, `autor_libro_enc`, `editorial_libro_enc`, `libro_hist__n`,
`libro_hist__avg`, ya en el catálogo) y 214 del lado del lector (`genero_lector`, `pais_lector`, y las
212 de `perfil_genero_actual`) — estas últimas son constantes para todos los candidatos de ese lector,
así que se asignan en bloque.

Si el lector no tiene fila en `perfil_genero_actual` (arranque en frío puro), completamos con el mismo
criterio de "sin historial" del resto del proyecto: `n` en 0, `avg`/`min`/`max` en `NaN` — XGBoost
también maneja *missing values* nativamente, igual que LightGBM.

**Diferencia real con `SR_Claude.ipynb`**: ahí había que reconvertir `genero_lector`/`pais_lector` al
`CategoricalDtype` exacto de entrenamiento o LightGBM tiraba `ValueError` en `predict()`. Acá XGBoost
matchea las categóricas por **valor**, no por código interno (lo verificamos antes de armar este
notebook), así que en rigor alcanzaría con que la columna sea `category`, sin que el conjunto de
categorías coincida exactamente. Igual dejamos el `astype` explícito, por consistencia con el resto del
proyecto y para no depender de ese detalle de implementación.

In [22]:
BOOK_FEATURE_COLS = ["anio_edicion_libro", "genero_libro", "autor_libro_enc", "editorial_libro_enc", "libro_hist__n", "libro_hist__avg"]

# perfil por default para un lector sin ninguna fila en perfil_genero_actual (arranque en frío
# puro): n en 0, avg/min/max en NaN -- mismo criterio de "sin historial" que el resto del proyecto
perfil_vacio = pd.Series(np.nan, index=GENERO_HIST_COLS)
perfil_vacio[[c for c in GENERO_HIST_COLS if c.startswith("genero_hist__n__")]] = 0


def ranking(id_lector, candidatos):
    """Predice el score de los libros en `candidatos` para `id_lector`. Devuelve (id_libro, score)."""
    X = candidatos[BOOK_FEATURE_COLS].copy()

    perfil = perfil_genero_actual.loc[id_lector] if id_lector in perfil_genero_actual.index else perfil_vacio
    bloque_perfil = pd.DataFrame(
        np.broadcast_to(perfil[GENERO_HIST_COLS].to_numpy(), (len(X), len(GENERO_HIST_COLS))),
        columns=GENERO_HIST_COLS, index=X.index,
    )
    X = pd.concat([X, bloque_perfil], axis=1)

    X["genero_lector"] = lectores_tabla.loc[id_lector, "genero_lector"]
    X["genero_lector"] = X["genero_lector"].astype(pd.CategoricalDtype(categories=cat_categories["genero_lector"]))
    X["pais_lector"] = lectores_tabla.loc[id_lector, "pais_lector"]
    X["pais_lector"] = X["pais_lector"].astype(pd.CategoricalDtype(categories=cat_categories["pais_lector"]))

    scores = final_model.predict(X[FEATURE_COLS])
    return candidatos["id_libro"].to_numpy(), scores


ids_demo, scores_demo = ranking(lector_demo, candidatos_demo)
pd.Series(scores_demo, index=ids_demo, name="pred_score").sort_values(ascending=False).head(5)

legado-a-mis-hijos           4.239709
40-anos-de-ciberactivismo    4.239709
la-palabra-como-arma         4.108413
el-juego-de-las-familias     3.520293
lluvia                       3.331266
Name: pred_score, dtype: float32

# Obtener predicciones

Generamos el top-20 para los mismos 3.890 lectores que usamos para medir NDCG — son los que ya tienen
un split de test armado, así que además de servir como "predicción real" del nuevo esquema, después
podemos comparar cualitativamente contra lo que cada uno efectivamente calificó alto. A esta velocidad
(dominada por el filtro de candidatos sobre el catálogo completo en `retrieval`), el lote entero tarda
menos de un minuto. Algunos lectores pueden quedar con menos de 20 candidatos, o ninguno, si sus
géneros/autores preferidos son muy poco comunes en el resto del catálogo — se cuentan y se listan en las
validaciones de sanidad, no rompen el resto de la corrida.

In [23]:
t0 = time.time()
resultados = []
lectores_sin_candidatos = []

for id_lector in df_test["id_lector"].unique():
    candidatos = retrieval(id_lector)
    if len(candidatos) == 0:
        lectores_sin_candidatos.append(id_lector)
        continue
    ids, scores = ranking(id_lector, candidatos)
    top20 = (
        pd.DataFrame({"id_lector": id_lector, "id_libro": ids, "pred_score": scores})
        .nlargest(20, "pred_score")
    )
    resultados.append(top20)

recomendaciones = pd.concat(resultados, ignore_index=True)
print(f"{df_test['id_lector'].nunique():,} lectores procesados en {time.time() - t0:.0f}s")
print(f"lectores sin ningún candidato: {len(lectores_sin_candidatos)}")
print(f"recomendaciones: {recomendaciones.shape}")

3,890 lectores procesados en 302s
lectores sin ningún candidato: 0
recomendaciones: (77800, 3)


## Validaciones de sanidad

In [24]:
filas_por_lector = recomendaciones.groupby("id_lector").size()
incompletos = filas_por_lector[filas_por_lector < 20]
print(f"lectores con menos de 20 recomendaciones: {len(incompletos)}")
if len(incompletos):
    print(incompletos.describe())

dup = recomendaciones.duplicated(subset=["id_lector", "id_libro"]).sum()
print(f"\npares (lector, libro) duplicados: {dup}")

ya_leidos_set = set(map(tuple, leidos[["id_lector", "id_libro"]].itertuples(index=False, name=None)))
recomendados_set = set(map(tuple, recomendaciones[["id_lector", "id_libro"]].itertuples(index=False, name=None)))
interseccion = recomendados_set & ya_leidos_set
print(f"recomendaciones que en realidad ya estaban leídas (debería ser 0): {len(interseccion)}")

assert dup == 0, "hay pares (lector, libro) duplicados"
assert len(interseccion) == 0, "se recomendó un libro ya leído"
print("\nOK: sin duplicados, sin libros ya leídos.")

lectores con menos de 20 recomendaciones: 0

pares (lector, libro) duplicados: 0
recomendaciones que en realidad ya estaban leídas (debería ser 0): 0

OK: sin duplicados, sin libros ya leídos.


## CSV final

In [25]:
salida = (
    recomendaciones.sort_values(["id_lector", "pred_score"], ascending=[True, False])
    [["id_lector", "id_libro"]]
    .reset_index(drop=True)
)
salida.to_csv(OUTPUT_CSV, index=False)
print(f"guardado en {OUTPUT_CSV} ({len(salida):,} filas, {salida['id_lector'].nunique():,} lectores)")
salida.head(10)

guardado en datos/sr_xgb_recomendaciones_top20.csv (77,800 filas, 3,890 lectores)


,id_lector,id_libro
0,05-03-1970,40-anos-de-ciberactivismo
1,05-03-1970,legado-a-mis-hijos
2,05-03-1970,la-palabra-como-arma
3,05-03-1970,el-juego-de-las-familias
4,05-03-1970,lluvia
5,05-03-1970,medio-vacio-o-medio-lleno
6,05-03-1970,atletico-de-madrid-una-pasion-una-gran-minoria
7,05-03-1970,ethel-y-ernest
8,05-03-1970,rojo-oscuro-sobre-azul
9,05-03-1970,hera


## Resumen y ejemplo

El top-5 recomendado sale de `retrieval()` (libros que el lector **no** leyó, ni en train ni en test);
lo que mejor calificó en test es, por definición, un conjunto disjunto de eso. No es una forma de medir
precisión (para eso está el NDCG de la sección de entrenamiento) — es solo un chequeo cualitativo de que
las recomendaciones tengan sentido con el gusto del lector (mismos géneros/autores que ya le gustaron).

In [26]:
conn = sqlite3.connect(DATABASE)
libros_info = pd.read_sql("SELECT id_libro, titulo, autor, genero FROM libros", conn)
conn.close()

lector_ejemplo = recomendaciones["id_lector"].drop_duplicates().sample(1, random_state=RANDOM_STATE).iloc[0]

top5_pred = (
    recomendaciones[recomendaciones["id_lector"] == lector_ejemplo]
    .sort_values("pred_score", ascending=False).head(5)
    .merge(libros_info, on="id_libro", how="left")
)
print(f"=== top 5 recomendado para {lector_ejemplo} ===")
print(top5_pred[["id_libro", "titulo", "autor", "genero", "pred_score"]].to_string(index=False))

mejor_calificado_test = (
    df_test[df_test["id_lector"] == lector_ejemplo]
    .sort_values("rating", ascending=False).head(5)
    .merge(libros_info, on="id_libro", how="left")
)
print(f"\n=== lo que {lector_ejemplo} mejor calificó en su propio test (no visto en train) ===")
print(mejor_calificado_test[["id_libro", "titulo", "autor", "genero", "rating"]].to_string(index=False))

=== top 5 recomendado para xanti ===
                 id_libro                    titulo                            autor                 genero  pred_score
40-anos-de-ciberactivismo 40 AÑOS DE CIBERACTIVISMO                        BRUNO, G.                 Ensayo    4.657664
       legado-a-mis-hijos        LEGADO A MIS HIJOS    BENITO DE BENITO, LUIS MIGUEL                 Ensayo    4.657664
     la-palabra-como-arma      LA PALABRA COMO ARMA                    GOLDMAN, EMMA                 Ensayo    4.462060
                   lluvia                    LLUVIA TALBOT, MARY M., y TALBOT, BRYAN Cómics, Novela Gráfica    3.922246
           ethel-y-ernest            ETHEL Y ERNEST                  BRIGGS, RAYMOND Cómics, Novela Gráfica    3.776357

=== lo que xanti mejor calificó en su propio test (no visto en train) ===
                                                                                   id_libro                                                                             